In [23]:
import os
import re

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from tqdm import tqdm

In [24]:
# === GPU 사용 설정 ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: Tesla T4


In [ ]:
num_classes = 10

class SafeDiskCachedCIFAR10(Dataset):
    def __init__(self, root='./data', train=True, cache_dir='./cifar10_32_cache'):
        self.cifar = torchvision.datasets.CIFAR10(root=root, train=train, download=True)
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        
        match = re.search(r'(\d+)_cache$', cache_dir)
        img_size = int(match.group(1)) if match else 128
        
        print(f"이미지 크기: {img_size}x{img_size}")

        self.transform = transforms.Compose([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
        
        self.cache_paths = []
        self.labels = []
        
        to_tensor = transforms.ToTensor()
        
        print(f"{'Train' if train else 'Test'} 데이터 캐싱 중...")
        
        for idx in tqdm(range(len(self.cifar)), desc="Caching"):
            cache_path = os.path.join(cache_dir, f"{idx}.pt")
            
            if not os.path.exists(cache_path):
                img, label = self.cifar[idx]
                img_tensor = to_tensor(img)
                torch.save(img_tensor, cache_path)
            
            self.cache_paths.append(cache_path)
            self.labels.append(self.cifar[idx][1])
        
        print(f"캐싱 완료! 총 {len(self.cache_paths)}장")

    def __len__(self):
        return len(self.cache_paths)

    def __getitem__(self, idx):
        img = torch.load(self.cache_paths[idx], weights_only=True)
        label = self.labels[idx]
        img = self.transform(img)
        return img, label


# ==================== 사용 ====================
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_32_cache') # 32x32로 그대로 유지(CIFAR10는 저해상도 이미지)
train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_128_cache')
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_160_cache') # 160x160일 때
# train_dataset = SafeDiskCachedCIFAR10(train=True, cache_dir='./cifar10_224_cache') # 224x224로 바꿀 때

test_dataset = SafeDiskCachedCIFAR10(train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

이미지 크기: 128x128
Train 데이터 캐싱 중 (단일 파일)...


Caching:  69%|██████▉   | 34588/50000 [00:05<00:02, 5865.18it/s]


KeyboardInterrupt: 

In [ ]:
base_model = models.resnet18(weights=None)
base_model = nn.Sequential(*list(base_model.children())[:-2])  # GlobalAveragePooling을 위해 마지막 두 레이어를 제거

In [ ]:
class CustomResNet50(nn.Module):
    def __init__(self, num_classes):
        super(CustomResNet50, self).__init__()
        self.base_model = base_model
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.base_model(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = CustomResNet50(num_classes=num_classes).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
scaler = GradScaler('cuda')

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        # === 중요: 데이터도 GPU로 이동 ===
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 1.4677076088794896
Epoch 2, Loss: 1.092612468959884
Epoch 3, Loss: 0.90038096471925
Epoch 4, Loss: 0.7388447091050127
Epoch 5, Loss: 0.6016732796750157
Epoch 6, Loss: 0.4885611442118521
Epoch 7, Loss: 0.382846647054815
Epoch 8, Loss: 0.3103592232947005
Epoch 9, Loss: 0.2533692600645282
Epoch 10, Loss: 0.21285442145146855


In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        # === 중요: 데이터도 GPU로 이동 ===
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 10.04%
